In [1]:
from typing import cast
import pandas as pd
import numpy as np
import bs4
import requests
from typing import Coroutine
import asyncio
import httpx
from typing import Any
SEMAFORO=asyncio.Semaphore(20)

In [2]:
url_principal:str='https://www.pisos.com/'
soup:bs4.BeautifulSoup=bs4.BeautifulSoup(requests.get(url=url_principal).text,'html.parser')
links:bs4.ResultSet=soup.find_all(class_='seo-box__location-link--level3')

In [3]:
opciones:list[str]=[opcion+"/" for opcion in ['venta','alquiler']]
tipos:list[str]=[tipo+"-" for tipo in ['pisos']]
tipos

['pisos-']

In [4]:
localidades:list[str]=[link['href'].split('-')[-1] for link in links]
localidades

['roquetas_de_mar/',
 'vera/',
 'almeria_capital/',
 'chiclana_de_la_frontera/',
 'jerez_de_la_frontera/',
 'cadiz_capital/',
 'cordoba_capital_zona_urbana/',
 'area_de_granada_granada_capital/',
 'almunecar/',
 'jaen_capital/',
 'marbella/',
 'estepona/',
 'mijas/',
 'sevilla_capital/',
 'dos_hermanas/',
 'zaragoza_capital/',
 'santander/',
 'salamanca_capital/',
 'valladolid_capital/',
 'albacete_capital_zona_urbana/',
 'barcelona_capital/',
 'sabadell/',
 'badalona/',
 'roses/',
 'castello_empuries/',
 'lloret_de_mar/',
 'calafell/',
 'madrid_capital_zona_urbana/',
 'torrevieja/',
 'orihuela/',
 'alicante_alacant/',
 'castello_de_la_plana/',
 'valencia_capital_zona_urbana/',
 'gandia/',
 'badajoz_capital/',
 'ourense_capital/',
 'vigo/',
 'palma_de_mallorca/',
 'calvia/',
 'las_palmas_de_gran_canaria/',
 'la_oliva/',
 'adeje/',
 'arona/',
 'logrono/',
 'san_sebastian_donostia/',
 'bilbao/',
 'oviedo/',
 'murcia_capital/',
 'los_alcazares/',
 'san_pedro_del_pinatar/']

In [5]:
urls:list[str]=[url_principal+opcion_tipo_localidad for opcion_tipo_localidad in [opcion+tipo_localidad for tipo_localidad in [tipo+localidad for localidad in localidades for tipo in tipos] for opcion in opciones]]
urls_venta:list[str]=urls[::2]
urls_alquiler:list[str]=urls[1::2]
urls

['https://www.pisos.com/venta/pisos-roquetas_de_mar/',
 'https://www.pisos.com/alquiler/pisos-roquetas_de_mar/',
 'https://www.pisos.com/venta/pisos-vera/',
 'https://www.pisos.com/alquiler/pisos-vera/',
 'https://www.pisos.com/venta/pisos-almeria_capital/',
 'https://www.pisos.com/alquiler/pisos-almeria_capital/',
 'https://www.pisos.com/venta/pisos-chiclana_de_la_frontera/',
 'https://www.pisos.com/alquiler/pisos-chiclana_de_la_frontera/',
 'https://www.pisos.com/venta/pisos-jerez_de_la_frontera/',
 'https://www.pisos.com/alquiler/pisos-jerez_de_la_frontera/',
 'https://www.pisos.com/venta/pisos-cadiz_capital/',
 'https://www.pisos.com/alquiler/pisos-cadiz_capital/',
 'https://www.pisos.com/venta/pisos-cordoba_capital_zona_urbana/',
 'https://www.pisos.com/alquiler/pisos-cordoba_capital_zona_urbana/',
 'https://www.pisos.com/venta/pisos-area_de_granada_granada_capital/',
 'https://www.pisos.com/alquiler/pisos-area_de_granada_granada_capital/',
 'https://www.pisos.com/venta/pisos-almu

In [6]:
async def consultar_n_paginas_opcion_localidad(id:int,u:str,client:httpx.AsyncClient)-> dict[int,int]:
    result:httpx.Response=await client.get(u)
    posible:bs4.Tag | None=bs4.BeautifulSoup(result.text,'html.parser').find(class_='grid__title')
    n:str='0'
    if title:=posible:
        r:str=title.find_all('span')[1].text
        if len(r)>0:
            n=r.split(' ')[0]
    resultados:int=int(n.replace('.',''))
    n_paginas_completas:int=resultados//30
    n_paginas:int=int(np.min([n_paginas_completas+(resultados>(30*n_paginas_completas)),100]))
    return {id:n_paginas}
async def sacar_n_paginas()->dict[int,int]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1)) as client:
        tareas:list[Coroutine[Any,Any,dict[int,int]]]=[consultar_n_paginas_opcion_localidad(id,u,client) for id,u in enumerate(urls)]
        tareas_completas:list[dict[int,int]]=await asyncio.gather(*tareas)
        return {k:v for d in tareas_completas for k,v in d.items()}
n_paginas:dict[int,int]=await sacar_n_paginas()
n_paginas

{0: 35,
 1: 4,
 2: 31,
 3: 4,
 4: 30,
 5: 13,
 6: 33,
 7: 3,
 8: 32,
 9: 2,
 10: 24,
 11: 4,
 12: 70,
 13: 7,
 14: 100,
 15: 27,
 16: 23,
 17: 8,
 18: 23,
 19: 6,
 20: 100,
 21: 22,
 22: 100,
 23: 8,
 24: 100,
 25: 4,
 26: 65,
 27: 25,
 28: 25,
 29: 2,
 30: 25,
 31: 5,
 32: 20,
 33: 7,
 34: 22,
 35: 27,
 36: 24,
 37: 3,
 38: 31,
 39: 4,
 40: 100,
 41: 35,
 42: 43,
 43: 1,
 44: 35,
 45: 2,
 46: 52,
 47: 1,
 48: 41,
 49: 1,
 50: 31,
 51: 1,
 52: 34,
 53: 1,
 54: 100,
 55: 100,
 56: 100,
 57: 8,
 58: 100,
 59: 6,
 60: 100,
 61: 15,
 62: 35,
 63: 3,
 64: 96,
 65: 45,
 66: 25,
 67: 9,
 68: 29,
 69: 5,
 70: 24,
 71: 6,
 72: 30,
 73: 5,
 74: 68,
 75: 10,
 76: 35,
 77: 6,
 78: 25,
 79: 9,
 80: 22,
 81: 1,
 82: 71,
 83: 5,
 84: 47,
 85: 5,
 86: 20,
 87: 1,
 88: 20,
 89: 6,
 90: 30,
 91: 10,
 92: 22,
 93: 13,
 94: 92,
 95: 14,
 96: 78,
 97: 2,
 98: 72,
 99: 2}

In [8]:
async def consultar_anuncios_cargados(client:httpx.AsyncClient,url:str,pagina:int)->list[str]:# junta todos los anuncios de una página de una localidad
    async with SEMAFORO:
        result:httpx.Response=await client.get(f"{url}{pagina}",timeout=10)
        ads=bs4.BeautifulSoup(result.text,'html.parser').find_all(class_='ad-preview')
        return [url_principal[:-1]+str(ad['data-lnk-href']) for ad in ads]
async def consultar_anuncios_url(client:httpx.AsyncClient,url:str)->list[str]: # junta todos los anuncios de una localidad
    tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_anuncios_cargados(client,url,i) for i in range(1,n_paginas[urls.index(url)]+1)]
    tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
    anuncios:list[str]=[u for lista in tareas_completas for u in lista]
    return anuncios
async def consultar_anuncios(lista_urls:list[str])->list[str]: # junta todos los anuncios
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1),limits=httpx.Limits(max_connections=20,max_keepalive_connections=20)) as client:
        tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_anuncios_url(client,url) for url in lista_urls]
        tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
        anuncios:list[str]=[u for lista in tareas_completas for u in lista]
    return anuncios

In [9]:
ventas_ads:list[str]=await consultar_anuncios(urls_venta)
ventas_ads

['https://www.pisos.com/comprar/piso-barrio_centro-60909552441_100500/',
 'https://www.pisos.com/comprar/piso-barrio_centro-61761982479_100500/',
 'https://www.pisos.com/comprar/chalet_adosado-barrio_centro-58377763502_993874/',
 'https://www.pisos.com/comprar/casa-barrio_centro-52518004110_993874/',
 'https://www.pisos.com/comprar/casa-barrio_centro-62567380224_517052/',
 'https://www.pisos.com/comprar/piso-roquetas_de_mar_el_parador_de_las_hortichuelas-65019858773_100200/',
 'https://www.pisos.com/comprar/casa_unifamiliar-urbanizacion_de_roquetas_las_marinas04740-65009934712_100200/',
 'https://www.pisos.com/comprar/piso-roquetas_de_mar-57548817160_100500/',
 'https://www.pisos.com/comprar/casa-la_romanilla_el_puerto-59208332405_108700/',
 'https://www.pisos.com/comprar/casa_unifamiliar-barrio_centro-59220479758_100200/',
 'https://www.pisos.com/comprar/piso-urbanizacion_de_roquetas_las_marinas04740-65907158721_100200/',
 'https://www.pisos.com/comprar/apartamento-urbanizacion_de_roq

In [10]:
len(ventas_ads)

74641

In [11]:
clases:list[str]=list(map(lambda x:str(x),np.unique(list(map(lambda x:x[x[:x.find('-')].rfind('/')+1:x.find('-')],ventas_ads)))))
clases

['apartamento',
 'atico',
 'casa',
 'casa_adosada',
 'casa_pareada',
 'casa_rustica',
 'casa_unifamiliar',
 'chalet',
 'chalet_adosado',
 'chalet_pareado',
 'chalet_rustico',
 'chalet_unifamiliar',
 'duplex',
 'estudio',
 'finca_rustica',
 'loft',
 'piso']

In [14]:
ventas_pisos:list[str]=list(filter(lambda x:'/piso' in x,ventas_ads))
ventas_pisos

['https://www.pisos.com/comprar/piso-barrio_centro-60909552441_100500/',
 'https://www.pisos.com/comprar/piso-barrio_centro-61761982479_100500/',
 'https://www.pisos.com/comprar/piso-roquetas_de_mar_el_parador_de_las_hortichuelas-65019858773_100200/',
 'https://www.pisos.com/comprar/piso-roquetas_de_mar-57548817160_100500/',
 'https://www.pisos.com/comprar/piso-urbanizacion_de_roquetas_las_marinas04740-65907158721_100200/',
 'https://www.pisos.com/comprar/piso-barrio_centro-65080463360_996739/',
 'https://www.pisos.com/comprar/piso-aguadulce_norte-58385576923_522833/',
 'https://www.pisos.com/comprar/piso-barrio_centro-63423769871_100500/',
 'https://www.pisos.com/comprar/piso-urbanizacion_de_roquetas_las_marinas04740-65021443765_998651/',
 'https://www.pisos.com/comprar/piso-nucleo_urbano_las_salinas04740-65891588733_100500/',
 'https://www.pisos.com/comprar/piso-av_juan_carlos_i_plaza_de_toros-64242172091_100500/',
 'https://www.pisos.com/comprar/piso-puerto_de_aguadulce-53338369614_

In [ ]:
async def consultar_caracteristicas(url:str,client:httpx.AsyncClient)->list[str]:
    async with SEMAFORO:
        response:httpx.Response=await client.get(url,timeout=10)
        features:bs4.ResultSet=bs4.BeautifulSoup(response.text,'html.parser').find_all(class_='features__label')
        return [feature.text for feature in features]

async def consultar_caracteristicas_importantes(urls:list[str],comunes:bool=False)->list[str]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1),limits=httpx.Limits(max_connections=20,max_keepalive_connections=20)) as client:
        tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_caracteristicas(vivienda,client) for vivienda in urls]
        tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
        caracteristicas:list[str]=[c for lista in tareas_completas for c in lista]
        repeticiones_necesarias:int=len(urls)
        if comunes:
            caracteristicas=list(filter(lambda x:caracteristicas.count(x)==repeticiones_necesarias,caracteristicas))
        return list(map(lambda x:str(x),np.unique(caracteristicas)))
# No hacer beautiful soup y guardar los htmls crudos
# Cambiar método de búsqueda de localidades

In [19]:
len(ventas_pisos)

38413

In [18]:
caracteristicas_pisos:list[str]=await consultar_caracteristicas_importantes(ventas_pisos[:20])
caracteristicas_pisos

['Adaptado a personas con movilidad reducida',
 'Agua',
 'Aire acondicionado',
 'Aire acondicionado: ',
 'Amueblado',
 'Antigüedad: ',
 'Armarios empotrados: ',
 'Ascensor',
 'Balcón',
 'Baños: ',
 'Calefacción',
 'Calefacción: ',
 'Carpintería exterior: ',
 'Carpintería interior',
 'Carpintería interior: ',
 'Cocina equipada',
 'Cocina equipada: ',
 'Comedor',
 'Conservación: ',
 'Exterior',
 'Garaje: ',
 'Gastos de comunidad: ',
 'Habitaciones: ',
 'Interior',
 'Jardín',
 'Jardín: ',
 'Lavadero',
 'Luz: ',
 'Orientación: ',
 'Piscina',
 'Piscina: ',
 'Planta: ',
 'Portero automático',
 'Puerta blindada',
 'Referencia: ',
 'Se aceptan mascotas',
 'Sistema de seguridad',
 'Sistema de seguridad: ',
 'Soleado',
 'Superficie construida: ',
 'Superficie útil: ',
 'Teléfono',
 'Terraza',
 'Tipo suelo: ',
 'Trastero',
 'Vidrios dobles']